In [1]:
import pandas as pd
import numpy as np
import re
import string
import pickle  
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score

In [2]:
df = pd.read_csv("amazon_vfl_reviews.csv")

df = df.dropna(subset=['review', 'rating'])

print(f"Dataset Loaded: {len(df)} rows")
print(df.iloc[0])

Dataset Loaded: 2776 rows
asin                                             B07W7CTLD1
name                Mamaearth-Onion-Growth-Control-Redensyl
date                                             2019-09-06
rating                                                    1
review    I bought this hair oil after viewing so many g...
Name: 0, dtype: object


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2776 entries, 0 to 2781
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   asin    2776 non-null   object
 1   name    2776 non-null   object
 2   date    2776 non-null   object
 3   rating  2776 non-null   int64 
 4   review  2776 non-null   object
dtypes: int64(1), object(4)
memory usage: 130.1+ KB


In [4]:
def clean_text(text):
    if not isinstance(text, str): return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # 3. Remove Punctuation (We keep numbers as they might indicate price/days)
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 4. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


print("Cleaning text... (This removes noise like HTML tags and symbols)")
df['clean_review'] = df['review'].apply(clean_text)

print("Text Cleaned.")
print(df[['review', 'clean_review']].head(2))

Cleaning text... (This removes noise like HTML tags and symbols)
Text Cleaned.
                                              review  \
0  I bought this hair oil after viewing so many g...   
1  Used This Mama Earth Newly Launched Onion Oil ...   

                                        clean_review  
0  i bought this hair oil after viewing so many g...  
1  used this mama earth newly launched onion oil ...  


In [5]:
# 1. Remove 3-star reviews (Neutral)
df_final = df[df['rating'] != 3].copy()

# 2. Create Labels: 1 = Positive (Happy), 0 = Negative (Sad)
# Logic: If rating > 3, it's a 1. Otherwise, it's a 0.
df_final['label'] = df_final['rating'].apply(lambda x: 1 if x > 3 else 0)

# Check the balance
print("Class Distribution:")
print(df_final['label'].value_counts())

Class Distribution:
label
1    1902
0     676
Name: count, dtype: int64


In [6]:
urgent_keywords = [
    "fake", "fraud", "scam", "cheat", "chor", "duplicate", "used product",
    "broken", "damaged", "tuta", "return", "refund", "waste", "garbage", 
    "bakwas", "useless", "worst", "ghatiya", "bad quality", "leak"
]

def check_urgency(text):
    text = str(text).lower()
    for word in urgent_keywords:
        if word in text:
            return True
    return False

# Apply logic
df_final['is_urgent'] = df_final['clean_review'].apply(check_urgency)

print(f"Urgent Cases Detected: {df_final['is_urgent'].sum()}")
print("--- Sample Urgent Review ---")
# Show one example to verify it works
if df_final['is_urgent'].sum() > 0:
    print(df_final[df_final['is_urgent'] == True]['review'].iloc[0])

Urgent Cases Detected: 350
--- Sample Urgent Review ---
I bought this hair oil after viewing so many good comments. But this product is not good enough.First of all it's Expensive...Second thing the amount of the product is low (half bottle) YES!The bottle is not completely filled with oil. If you cheating on your customers #Mamaearth trust me on this you can't fool people more than once. Now I know that your Brand is not good enough. I am not going to buy any product from your Brand again.Thumbs down for mamaearth onion oil !!


In [7]:
# X = Input (The clean text)
# y = Output (The 1/0 label)
X = df_final['clean_review']
y = df_final['label']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {len(X_train)} reviews")
print(f"Testing on {len(X_test)} reviews")

Training on 2062 reviews
Testing on 516 reviews


In [8]:
# 1. Build the Pipeline
model_pipeline = make_pipeline(
    TfidfVectorizer(
        max_features=5000,  # Keep only top 5k important words
        ngram_range=(1,2)   # Capture context like "not good"
    ),
    LinearSVC(
        random_state=42,
        class_weight='balanced' # Fixes the imbalance automatically
    )
)


model_pipeline.fit(X_train, y_train)

print("Model Trained Successfully.")

Model Trained Successfully.


In [9]:
predictions = model_pipeline.predict(X_test)

acc_score = accuracy_score(y_test, predictions)
print(f"Accuracy Score: {acc_score * 100:.2f}%")

# 3. Detailed Report
print("\n--- Classification Report ---")
# Target Names: 0 = Negative, 1 = Positive
print(classification_report(y_test, predictions, target_names=['Negative', 'Positive']))

Accuracy Score: 96.51%

--- Classification Report ---
              precision    recall  f1-score   support

    Negative       0.97      0.92      0.94       155
    Positive       0.96      0.99      0.98       361

    accuracy                           0.97       516
   macro avg       0.97      0.95      0.96       516
weighted avg       0.97      0.97      0.96       516



In [10]:
def predict_review(text):
    # 1. Clean the text (using the function we defined earlier)
    clean_text_input = clean_text(text)
    
    # 2. Predict Sentiment (1=Positive, 0=Negative)
    # .predict() returns an array, so we take [0]
    prediction = model_pipeline.predict([clean_text_input])[0]
    sentiment = "Positive" if prediction == 1 else "Negative"
    
    # 3. Check Urgency (Rule-Based)
    is_urgent = check_urgency(text)
    priority = "URGENT" if is_urgent else "Normal"
    
    return f"Review: '{text}'\n   -> Sentiment: {sentiment}\n   -> Priority:  {priority}\n"

# Test with your own Hinglish examples!
print(predict_review("Product mast hai but delivery late thi"))
print(predict_review("Bakwas product broken screen returned it"))
print(predict_review("Simply awesome, loved it"))
print(predict_review("chor seller fake item sent"))

Review: 'Product mast hai but delivery late thi'
   -> Sentiment: Positive
   -> Priority:  Normal

Review: 'Bakwas product broken screen returned it'
   -> Sentiment: Negative
   -> Priority:  URGENT

Review: 'Simply awesome, loved it'
   -> Sentiment: Positive
   -> Priority:  Normal

Review: 'chor seller fake item sent'
   -> Sentiment: Negative
   -> Priority:  URGENT



In [11]:
model_filename = "cits.pkl"

with open(model_filename, "wb") as f:
    pickle.dump(model_pipeline, f)

print(f"Model saved as '{model_filename}'.")
print("Phase 1 Complete! You are ready to build the API.")

Model saved as 'cits.pkl'.
Phase 1 Complete! You are ready to build the API.


In [16]:
df_final.info()
df_final.shape

<class 'pandas.core.frame.DataFrame'>
Index: 2578 entries, 0 to 2781
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   asin          2578 non-null   object
 1   name          2578 non-null   object
 2   date          2578 non-null   object
 3   rating        2578 non-null   int64 
 4   review        2578 non-null   object
 5   clean_review  2578 non-null   object
 6   label         2578 non-null   int64 
 7   is_urgent     2578 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 163.6+ KB


(2578, 8)

In [17]:
df_final.head()

,asin,name,date,rating,review,clean_review,label,is_urgent
0,B07W7CTLD1,Mamaearth-Onion-Growth-Control-Redensyl,2019-09-06,1,I bought this hair oil after viewing so many g...,i bought this hair oil after viewing so many g...,0,True
1,B07W7CTLD1,Mamaearth-Onion-Growth-Control-Redensyl,2019-08-14,5,Used This Mama Earth Newly Launched Onion Oil ...,used this mama earth newly launched onion oil ...,1,False
2,B07W7CTLD1,Mamaearth-Onion-Growth-Control-Redensyl,2019-10-19,1,So bad product...My hair falling increase too ...,so bad productmy hair falling increase too muc...,0,False
3,B07W7CTLD1,Mamaearth-Onion-Growth-Control-Redensyl,2019-09-16,1,Product just smells similar to navarathna hair...,product just smells similar to navarathna hair...,0,True
4,B07W7CTLD1,Mamaearth-Onion-Growth-Control-Redensyl,2019-08-18,5,I have been trying different onion oil for my ...,i have been trying different onion oil for my ...,1,False
